# Step 1 Inventory: GOSH
Profiles every file at the site (columns, types, rows, ID and date formats) and records what later steps must be told: candidate keys, files that look like copies of one another, free-text columns, and years where a file's rows collapse. Nothing is changed. Outputs go to `derived/` and are counts, shapes and column names only.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import inventory as I

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
site = "GOSH"                      # "RBH" | "SMH" | "GOSH"
demo = "PICANet_ALL_patients_SR_011024_projectID_eprweights.csv" # GOSH: "PICANet_ALL_patients_SR_011024_projectID_eprweights.csv"
exclude = None                    # GOSH: r"lab_components"
recursive, n_jobs = True, 3      # GOSH: True, 3 (files sit in subfolders)

base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
derived_dir = open_picu_dir / "derived"
reports_dir = open_picu_dir / "reports"site_dir = base / site

inv = I.inventory_site(site_dir, recursive=recursive, n_jobs=n_jobs)
I.save_inventory(inv, derived_dir / "inventory" / f"{site.lower()}_inventory.pkl")
inv["summary"]

## 1. Structure
Files with identical columns are grouped (one file per variable is common at GOSH). Candidate keys are every ID-like column; "unique per row" tells you which could identify a row. ID overlap is the share of each file's IDs that appear in the base file.

In [ ]:
I.schema_groups(inv)

In [ ]:
I.id_overlap(inv, base=demo)

## 2. Dates and eras
Format, range and resolution of every date column, then rows per calendar year by each file's main time column. `find_gaps` lists years below half of the file's usual level. These are leads to explain (system changeover, outage, pandemic), not errors. `rows_by_year` rereads one column per file, so allow a few minutes at GOSH.

In [ ]:
inv["dates"]

In [ ]:
by_year = I.rows_by_year(inv)
by_year

In [ ]:
I.find_gaps(by_year)

## 3. Privacy flags and look-alike files
Columns flagged as free text or direct identifiers are carried forward: no later step may release a column with one of these names without a decision. Look-alike files are candidates to settle at step 2.

In [ ]:
I.identifier_audit(inv)

In [ ]:
I.lookalike_files(inv)

## 4. Decisions and notes
The only place judgements are typed. `rejected` lists files that must not be used and why. `notes` are short statements, with no values or IDs, that later steps should know.

In [ ]:
# RBH
# rejected = {"ai_picu_piccanet_old.csv": "old PICANet file; not used (manifest rule 2)"}
# notes = ["studyID = prefix (NIC/NND) : PATIENT_RK : SAS-second admission time; 5 stays appear under both prefixes",
#          "NND rows admitted before 2010 mostly have no vitals (7,917 empty studyIDs)",
#          "demographics holds 14,797 studyIDs, 1,139 admissions have more than one studyID"]

# SMH
# rejected = {}
# notes = ["encounterId is the stay key; UniquePtID is not equal between demographics and piccanet, so do not link on it",
#          "large exact-duplicate row counts in vitals, vent, airway, io (see summary)"]

# GOSH
rejected = {"Archived/picanet_file_with_mapping.csv": "unreliable discharge status; not used",
            "PICANet_All_patients_SR_011024_projectID.csv": "identical to the eprweights file; eprweights used"}
notes = ["links by project_id; admission_identifier on about 56% of HR rows only",
         "Epic era from 2019-04-20: see GOSH_EPIC_ERA.md"]

In [ ]:
paths = I.finalise(inv, site, reports_dir, base=demo, by_year=by_year,
                   rejected=rejected, excluded=exclude, notes=notes)
paths

In [ ]:
paths = I.finalise(inv, site, reports_dir, base=demo, by_year=by_year,
                   rejected=rejected, excluded=kwargs.get("exclude"), notes=notes)
paths

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import pandas as pd
import inventory as I
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
gosh_dir = base / "GOSH"

I.list_files(gosh_dir)

In [ ]:
gosh = I.inventory_site(gosh_dir, recursive=True, exclude=r"lab_components", n_jobs=3)
I.save_inventory(gosh, open_picu_dir/"derived"/"gosh_inventory.pkl")
gosh["summary"].sort_values("seconds", ascending=False).head(10)

In [ ]:
raw = I.inventory_site(gosh_dir/"raw_lab_values", recursive=False,
                       exclude=r"^lab_variables_(?!INR|Glucose).*lab_components")

In [ ]:
base = "PICANet_All_patients_SR_011024_projectID.csv"
pic = ["Archived/picanet_file_with_mapping.csv", base,
       "PICANet_ALL_patients_SR_011024_projectID_eprweights.csv",
       "gosh_mech_vent_31March2026/final_gosh_dataset.parquet"]

d = gosh["dates"].copy()
d["earliest"] = pd.to_datetime(d["earliest"])
d["latest"] = pd.to_datetime(d["latest"])
dsum = d.groupby("column").agg(files=("file", "count"), earliest=("earliest", "min"), latest=("latest", "max"))

parts = ["# 1. Schema groups", I.df_text(I.schema_groups(gosh), index=False)]
for fn in pic:
    parts += [f"# 2. Profile: {fn}", I.df_text(gosh["profiles"][fn], index=False)]
parts += [f"# 3. ID overlap with {base}", I.df_text(I.id_overlap(gosh, base=base), index=False)]
parts += ["# 4. Dates by column", I.df_text(dsum)]
parts += ["# 5. Logs and notes (each cut at 4,000 characters)"]
for p in sorted(gosh_dir.rglob("*.txt")):
    text = p.read_text(errors="replace")
    parts += [f"## {p.relative_to(gosh_dir).as_posix()} ({len(text)} characters)", text[:4000]]

out = open_picu_dir/"derived"/"gosh_for_claude.md"
out.write_text("\n\n".join(parts))
print(out, round(out.stat().st_size / 1e3), "KB")

In [ ]:
import importlib, gosh_links as GL
importlib.reload(GL)

GL.candidates(gosh_dir, [r"picanet", r"heart|pulse|^HR|/HR", r"\.parquet$"])

In [ ]:
gosh_dir


In [ ]:
gosh_links = GL.check_links(
    gosh_dir,
    picanet=gosh_dir/"PICANet_All_patients_SR_011024_projectID.csv",
    picanet_epr=gosh_dir/"PICANet_ALL_patients_SR_011024_projectID_eprweights.csv",
    archived=gosh_dir/"Archived/picanet_file_with_mapping.csv",
    hr_file=gosh_dir/ "merged_values" / "HR_merged.csv",
    parquet=gosh_dir/"gosh_mech_vent_31March2026/final_gosh_dataset.parquet",
)

In [ ]:
importlib.reload(GL)
GL.check_windows(
    gosh_dir,
    archived="Archived/picanet_file_with_mapping.csv",
    parquet="gosh_mech_vent_31March2026/final_gosh_dataset.parquet",
    links=gosh_links,
)

In [ ]:
importlib.reload(GL)
GL.candidates(gosh_dir, [r"merged_values/.*merged\.csv$"])

In [ ]:
vit = GL.vitals_coverage(
    gosh_dir,
    archived="Archived/picanet_file_with_mapping.csv",
    files=[
        "merged_values/HR_merged.csv",
        "merged_values/PulseRate_merged.csv",
        "merged_values/RespiratoryRate_merged.csv",
        "merged_values/SpO2_merged.csv",
        "merged_values/Temperature_merged.csv",
        "merged_values/Invasive_systolic_BP_merged.csv",
        "merged_values/Invasive_diastolic_BP_merged.csv",
        "merged_values/Invasive_mean_BP_merged.csv",
        "merged_values/Non-Invasive_systolic_BP_merged.csv",
        "merged_values/Non-Invasive_diastolic_BP_merged.csv",
        "merged_values/Non-Invasive_mean_BP_merged.csv",
    ],
)

In [ ]:
importlib.reload(GL)
cmp = GL.compare_picanet(
    gosh_dir,
    picanet="PICANet_All_patients_SR_011024_projectID.csv",
    picanet_epr="PICANet_ALL_patients_SR_011024_projectID_eprweights.csv",
    archived="Archived/picanet_file_with_mapping.csv",
)

In [ ]:
importlib.reload(GL)
oc = GL.compare_outcomes(
    gosh_dir,
    picanet="PICANet_All_patients_SR_011024_projectID.csv",
    archived="Archived/picanet_file_with_mapping.csv",
)

In [ ]:
importlib.reload(GL)
st = GL.compare_status(
    gosh_dir,
    picanet="PICANet_All_patients_SR_011024_projectID.csv",
    archived="Archived/picanet_file_with_mapping.csv",
)

In [ ]:
importlib.reload(GL)
vit_main = GL.vitals_coverage(
    gosh_dir,
    archived="PICANet_ALL_patients_SR_011024_projectID_eprweights.csv",
    start_col="AdDateTime",
    end_col="UnitDisDateTime",
    files=[
        "merged_values/HR_merged.csv",
        "merged_values/PulseRate_merged.csv",
        "merged_values/RespiratoryRate_merged.csv",
        "merged_values/SpO2_merged.csv",
        "merged_values/Temperature_merged.csv",
        "merged_values/Invasive_systolic_BP_merged.csv",
        "merged_values/Invasive_diastolic_BP_merged.csv",
        "merged_values/Invasive_mean_BP_merged.csv",
        "merged_values/Non-Invasive_systolic_BP_merged.csv",
        "merged_values/Non-Invasive_diastolic_BP_merged.csv",
        "merged_values/Non-Invasive_mean_BP_merged.csv",
    ],
)

In [ ]:
dates_gosh = I.date_audit(gosh, gosh_dir)
I.write_report(
    gosh, "GOSH",
    open_picu_dir / "reports" / "GOSH_inventory.md",
    base="PICANet_All_patients_SR_011024_projectID.csv",
    dates=dates_gosh,
)